# 3 — Previsão de Churn com Machine Learning

**Objetivo:** Construir modelos de classificação (Random Forest e XGBoost) para prever se um cliente tem alta probabilidade de churn (`Exited = 1`), avaliar com Accuracy, Curva ROC, AUC e Matriz de Confusão.

**Dataset:** `datasets/processed/bank_churn.db` (modelo dimensional SQLite — 10.000 registros)

**Premissas:**
- Dados provenientes do modelo dimensional já validado
- Variável alvo: `exited` (0 = Retido, 1 = Churned)
- Split treino/teste: 80/20 com estratificação
- `random_state=42` em todo modelo
- Métricas de avaliação: Accuracy, ROC-AUC, Curva ROC, Matriz de Confusão
- Gráficos exclusivamente com Plotly (paleta BI Dark Green)

---
## 1. Setup e Conexão

## 1.1 Imports e constantes

In [1]:
# Imports de bibliotecas e constantes de layout padrão
import sqlite3
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)

from xgboost import XGBClassifier

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Layout padrão — Paleta 1 (BI Dark Green)
LAYOUT_PADRAO: dict = dict(
    template='plotly_dark',
    paper_bgcolor='#1A1A1A',
    plot_bgcolor='#0F2015',
    font=dict(color='#CCCCCC', size=12, family='Arial'),
    title_font=dict(color='#39FF5A', size=16),
)

COLOR_SEQUENCE: list[str] = ['#39FF5A', '#004D54', '#2ECC71', '#CCCCCC']
RANDOM_STATE: int = 42
IMAGES_PATH: Path = Path('..') / 'datasets' / 'images'

print('Setup concluído.')

Setup concluído.


## 1.2 Conexão e carregamento dos dados

In [2]:
# Conexão com o banco dimensional e carregamento da base analítica
DB_PATH: Path = Path('..') / 'datasets' / 'processed' / 'bank_churn.db'
conn: sqlite3.Connection = sqlite3.connect(str(DB_PATH))

SQL_BASE: str = """
SELECT
    c.customer_id,
    c.credit_score,
    c.gender,
    c.age,
    c.tenure,
    c.estimated_salary,
    g.geography_name   AS geography,
    f.balance,
    f.num_of_products,
    f.has_credit_card,
    f.is_active_member,
    f.exited
FROM fact_bank_churn f
INNER JOIN dim_customers  c ON f.customer_id  = c.customer_id
INNER JOIN dim_geography  g ON f.geography_id = g.geography_id
"""

df_bank_churn: pd.DataFrame = pd.read_sql_query(SQL_BASE, conn)
conn.close()

print(f'Base carregada: {df_bank_churn.shape[0]:,} registros × {df_bank_churn.shape[1]} colunas')
print(f'Distribuição do target (exited):')
print(df_bank_churn['exited'].value_counts().to_string())
print(f'\nTaxa de churn: {df_bank_churn["exited"].mean() * 100:.2f}%')

Base carregada: 10,000 registros × 12 colunas
Distribuição do target (exited):
exited
0    7963
1    2037

Taxa de churn: 20.37%


---
## 2. Preparação dos Dados

## 2.1 Feature engineering e encoding

In [3]:
# Preparar features: encoding de variáveis categóricas e seleção de colunas
df_model: pd.DataFrame = df_bank_churn.drop(columns=['customer_id']).copy()

# Label Encoding para gender (binário)
le_gender: LabelEncoder = LabelEncoder()
df_model['gender_encoded'] = le_gender.fit_transform(df_model['gender'])
print(f'Gender encoding: {dict(zip(le_gender.classes_, le_gender.transform(le_gender.classes_)))}')

# One-Hot Encoding para geography (3 categorias)
df_model = pd.get_dummies(df_model, columns=['geography'], prefix='geo', dtype=int)

# Remover coluna original gender
df_model = df_model.drop(columns=['gender'])

print(f'\nFeatures finais ({df_model.shape[1] - 1} features + 1 target):')
print([col for col in df_model.columns if col != 'exited'])
df_model.head()

Gender encoding: {'Female': np.int64(0), 'Male': np.int64(1)}

Features finais (12 features + 1 target):
['credit_score', 'age', 'tenure', 'estimated_salary', 'balance', 'num_of_products', 'has_credit_card', 'is_active_member', 'gender_encoded', 'geo_France', 'geo_Germany', 'geo_Spain']


,credit_score,age,tenure,estimated_salary,balance,num_of_products,has_credit_card,is_active_member,exited,gender_encoded,geo_France,geo_Germany,geo_Spain
0,698,39,9,90212.38,161993.89,1,0,0,0,0,0,0,1
1,612,35,1,83256.26,0.00,1,1,1,1,1,0,0,1
2,601,47,1,96517.97,64430.06,2,0,1,0,1,1,0,0
3,627,30,6,188258.49,57809.32,1,1,0,0,0,0,1,0
4,745,48,10,74510.65,96048.55,1,1,0,0,1,0,1,0


## 2.2 Separação treino / teste

In [4]:
# Separar features (X) e target (y), split 80/20 estratificado
FEATURES: list[str] = [col for col in df_model.columns if col != 'exited']
TARGET: str = 'exited'

X: pd.DataFrame = df_model[FEATURES]
y: pd.Series = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f'Treino: {X_train.shape[0]:,} registros ({y_train.mean() * 100:.2f}% churn)')
print(f'Teste:  {X_test.shape[0]:,} registros ({y_test.mean() * 100:.2f}% churn)')

Treino: 8,000 registros (20.38% churn)
Teste:  2,000 registros (20.35% churn)


## 2.3 Padronização (StandardScaler)

In [5]:
# Padronizar features numéricas para melhor convergência dos modelos
scaler: StandardScaler = StandardScaler()

X_train_scaled: np.ndarray = scaler.fit_transform(X_train)
X_test_scaled: np.ndarray = scaler.transform(X_test)

print(f'Scaler ajustado em {X_train_scaled.shape[1]} features')
print(f'Médias pós-escala (treino): min={X_train_scaled.mean(axis=0).min():.4f}, max={X_train_scaled.mean(axis=0).max():.4f}')

Scaler ajustado em 12 features
Médias pós-escala (treino): min=-0.0000, max=0.0000


---
## 3. Modelo 1 — Random Forest

## 3.1 Treinamento

In [6]:
# Treinar Random Forest Classifier
rf_model: RandomForestClassifier = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

rf_model.fit(X_train_scaled, y_train)

# Previsões
y_pred_rf: np.ndarray = rf_model.predict(X_test_scaled)
y_proba_rf: np.ndarray = rf_model.predict_proba(X_test_scaled)[:, 1]

print('Random Forest treinado com sucesso.')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_rf) * 100:.2f}%')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_proba_rf):.4f}')

Random Forest treinado com sucesso.
Accuracy:  84.85%
ROC-AUC:   0.8798


## 3.2 Avaliação — Random Forest

In [7]:
# Classification report detalhado — Random Forest
print('=== Classification Report — Random Forest ===')
print(classification_report(y_test, y_pred_rf, target_names=['Retido (0)', 'Churned (1)']))

=== Classification Report — Random Forest ===
              precision    recall  f1-score   support

  Retido (0)       0.92      0.88      0.90      1593
 Churned (1)       0.61      0.71      0.66       407

    accuracy                           0.85      2000
   macro avg       0.77      0.80      0.78      2000
weighted avg       0.86      0.85      0.85      2000



In [8]:
# Matriz de confusão — Random Forest
cm_rf: np.ndarray = confusion_matrix(y_test, y_pred_rf)

labels_cm: list[str] = ['Retido (0)', 'Churned (1)']

fig_cm_rf: go.Figure = go.Figure(data=go.Heatmap(
    z=cm_rf,
    x=labels_cm,
    y=labels_cm,
    text=cm_rf,
    texttemplate='%{text}',
    textfont=dict(size=18, color='white'),
    colorscale=[[0, '#0F2015'], [0.5, '#004D54'], [1, '#39FF5A']],
    showscale=False,
    hovertemplate='Real: %{y}<br>Previsto: %{x}<br>Qtd: %{z}<extra></extra>',
))

fig_cm_rf.update_layout(
    **LAYOUT_PADRAO,
    title='Matriz de Confusão — Random Forest',
    xaxis_title='Previsto',
    yaxis_title='Real',
    yaxis=dict(autorange='reversed'),
    width=500,
    height=450,
)
fig_cm_rf.write_html(str(IMAGES_PATH / '10_cm_random_forest.html'))
fig_cm_rf.show()

## 3.3 Feature Importance — Random Forest

In [9]:
# Importância das features — Random Forest
df_importance_rf: pd.DataFrame = pd.DataFrame({
    'feature': FEATURES,
    'importance': rf_model.feature_importances_,
}).sort_values('importance', ascending=True)

fig_imp_rf: go.Figure = px.bar(
    df_importance_rf,
    x='importance',
    y='feature',
    orientation='h',
    title='Importância das Features — Random Forest',
    labels={'importance': 'Importância', 'feature': 'Feature'},
    color_discrete_sequence=['#39FF5A'],
)
fig_imp_rf.update_layout(**LAYOUT_PADRAO)
fig_imp_rf.write_html(str(IMAGES_PATH / '11_feature_importance_rf.html'))
fig_imp_rf.show()

### Insights — Random Forest

- **Accuracy: 84.85%** — desempenho sólido na classificação geral.
- **ROC-AUC: 0.8798** — boa capacidade de separação entre retidos e churned.
- **Recall de churn: 71.25%** — identifica 290 dos 407 churned reais, mas **117 falsos negativos** (clientes que saem sem ser detectados).
- **Features mais importantes:** `age`, `num_of_products`, `balance`, `is_active_member` dominam as decisões — alinhado com análises exploratórias anteriores.
- `class_weight='balanced'` ajudou a mitigar o desbalanceamento (80/20).

---
## 4. Modelo 2 — XGBoost

## 4.1 Treinamento

In [10]:
# Calcular scale_pos_weight para lidar com desbalanceamento
n_retidos: int = int((y_train == 0).sum())
n_churned: int = int((y_train == 1).sum())
scale_pos_weight: float = n_retidos / n_churned

print(f'Classes no treino: Retidos={n_retidos:,}, Churned={n_churned:,}')
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

# Treinar XGBoost Classifier
xgb_model: XGBClassifier = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    eval_metric='logloss',
    n_jobs=-1,
)

xgb_model.fit(X_train_scaled, y_train)

# Previsões
y_pred_xgb: np.ndarray = xgb_model.predict(X_test_scaled)
y_proba_xgb: np.ndarray = xgb_model.predict_proba(X_test_scaled)[:, 1]

print(f'\nXGBoost treinado com sucesso.')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_xgb) * 100:.2f}%')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_proba_xgb):.4f}')

Classes no treino: Retidos=6,370, Churned=1,630
scale_pos_weight: 3.91

XGBoost treinado com sucesso.
Accuracy:  83.20%
ROC-AUC:   0.8711


## 4.2 Avaliação — XGBoost

In [11]:
# Classification report detalhado — XGBoost
print('=== Classification Report — XGBoost ===')
print(classification_report(y_test, y_pred_xgb, target_names=['Retido (0)', 'Churned (1)']))

=== Classification Report — XGBoost ===
              precision    recall  f1-score   support

  Retido (0)       0.92      0.87      0.89      1593
 Churned (1)       0.57      0.69      0.63       407

    accuracy                           0.83      2000
   macro avg       0.74      0.78      0.76      2000
weighted avg       0.85      0.83      0.84      2000



In [12]:
# Matriz de confusão — XGBoost
cm_xgb: np.ndarray = confusion_matrix(y_test, y_pred_xgb)

fig_cm_xgb: go.Figure = go.Figure(data=go.Heatmap(
    z=cm_xgb,
    x=labels_cm,
    y=labels_cm,
    text=cm_xgb,
    texttemplate='%{text}',
    textfont=dict(size=18, color='white'),
    colorscale=[[0, '#0F2015'], [0.5, '#004D54'], [1, '#39FF5A']],
    showscale=False,
    hovertemplate='Real: %{y}<br>Previsto: %{x}<br>Qtd: %{z}<extra></extra>',
))

fig_cm_xgb.update_layout(
    **LAYOUT_PADRAO,
    title='Matriz de Confusão — XGBoost',
    xaxis_title='Previsto',
    yaxis_title='Real',
    yaxis=dict(autorange='reversed'),
    width=500,
    height=450,
)
fig_cm_xgb.write_html(str(IMAGES_PATH / '12_cm_xgboost.html'))
fig_cm_xgb.show()

## 4.3 Feature Importance — XGBoost

In [13]:
# Importância das features — XGBoost
df_importance_xgb: pd.DataFrame = pd.DataFrame({
    'feature': FEATURES,
    'importance': xgb_model.feature_importances_,
}).sort_values('importance', ascending=True)

fig_imp_xgb: go.Figure = px.bar(
    df_importance_xgb,
    x='importance',
    y='feature',
    orientation='h',
    title='Importância das Features — XGBoost',
    labels={'importance': 'Importância', 'feature': 'Feature'},
    color_discrete_sequence=['#39FF5A'],
)
fig_imp_xgb.update_layout(**LAYOUT_PADRAO)
fig_imp_xgb.write_html(str(IMAGES_PATH / '13_feature_importance_xgb.html'))
fig_imp_xgb.show()

### Insights — XGBoost

- **Accuracy: 83.20%** — ligeiramente inferior ao Random Forest (-1.65pp).
- **ROC-AUC: 0.8711** — capacidade discriminativa próxima ao RF, mas levemente abaixo.
- **Recall de churn: 69.29%** — identifica 282 churned, com **125 falsos negativos** (8 a mais que o RF).
- **Precision churn: 57.20%** — mais alarmes falsos que o RF (211 vs 186 falsos positivos).
- Features de maior importância seguem o mesmo padrão: `age`, `num_of_products` e `balance`.

---
## 5. Comparação dos Modelos

## 5.1 Tabela comparativa

In [14]:
# Tabela comparativa de métricas entre os dois modelos
def calculate_metrics(y_true: pd.Series, y_pred: np.ndarray, y_proba: np.ndarray, model_name: str) -> dict:
    """
    Calcula métricas de classificação para um modelo.

    Args:
        y_true: Valores reais do target.
        y_pred: Previsões binárias.
        y_proba: Probabilidades da classe positiva.
        model_name: Nome do modelo.

    Returns:
        Dicionário com métricas do modelo.
    """
    cm: np.ndarray = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    return {
        'modelo': model_name,
        'accuracy': round(accuracy_score(y_true, y_pred) * 100, 2),
        'roc_auc': round(roc_auc_score(y_true, y_proba), 4),
        'true_positives': tp,
        'false_negatives': fn,
        'true_negatives': tn,
        'false_positives': fp,
        'recall_churn': round(tp / (tp + fn) * 100, 2),
        'precision_churn': round(tp / (tp + fp) * 100, 2),
    }


metrics_rf: dict = calculate_metrics(y_test, y_pred_rf, y_proba_rf, 'Random Forest')
metrics_xgb: dict = calculate_metrics(y_test, y_pred_xgb, y_proba_xgb, 'XGBoost')

df_comparacao: pd.DataFrame = pd.DataFrame([metrics_rf, metrics_xgb])
df_comparacao

,modelo,accuracy,roc_auc,true_positives,false_negatives,true_negatives,false_positives,recall_churn,precision_churn
0,Random Forest,84.85,0.8798,290,117,1407,186,71.25,60.92
1,XGBoost,83.20,0.8711,282,125,1382,211,69.29,57.20


## 5.2 Comparação de Accuracy

In [15]:
# Gráfico de barras — Accuracy comparativa
fig_acc: go.Figure = go.Figure()

fig_acc.add_trace(go.Bar(
    x=[metrics_rf['modelo']],
    y=[metrics_rf['accuracy']],
    name='Random Forest',
    marker_color='#39FF5A',
    text=[f"{metrics_rf['accuracy']}%"],
    textposition='outside',
    textfont=dict(size=16),
    width=0.35,
))

fig_acc.add_trace(go.Bar(
    x=[metrics_xgb['modelo']],
    y=[metrics_xgb['accuracy']],
    name='XGBoost',
    marker_color='#004D54',
    text=[f"{metrics_xgb['accuracy']}%"],
    textposition='outside',
    textfont=dict(size=16),
    width=0.35,
))

fig_acc.update_layout(
    **LAYOUT_PADRAO,
    title='Comparação de Accuracy — Random Forest vs XGBoost',
    yaxis_title='Accuracy (%)',
    yaxis=dict(range=[0, 100]),
    showlegend=False,
)
fig_acc.write_html(str(IMAGES_PATH / '14_accuracy_comparison.html'))
fig_acc.show()

## 5.3 Curva ROC — Comparação

In [16]:
# Curva ROC comparativa — Random Forest vs XGBoost
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba_rf)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_proba_xgb)

auc_rf: float = roc_auc_score(y_test, y_proba_rf)
auc_xgb: float = roc_auc_score(y_test, y_proba_xgb)

fig_roc: go.Figure = go.Figure()

# Random Forest
fig_roc.add_trace(go.Scatter(
    x=fpr_rf, y=tpr_rf,
    mode='lines',
    name=f'Random Forest (AUC = {auc_rf:.4f})',
    line=dict(color='#39FF5A', width=2.5),
))

# XGBoost
fig_roc.add_trace(go.Scatter(
    x=fpr_xgb, y=tpr_xgb,
    mode='lines',
    name=f'XGBoost (AUC = {auc_xgb:.4f})',
    line=dict(color='#2ECC71', width=2.5),
))

# Linha diagonal (classificador aleatório)
fig_roc.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines',
    name='Aleatório (AUC = 0.50)',
    line=dict(color='#CCCCCC', width=1, dash='dash'),
))

fig_roc.update_layout(
    **LAYOUT_PADRAO,
    title='Curva ROC — Random Forest vs XGBoost',
    xaxis_title='Taxa de Falsos Positivos (FPR)',
    yaxis_title='Taxa de Verdadeiros Positivos (TPR)',
    xaxis=dict(range=[0, 1]),
    yaxis=dict(range=[0, 1.05]),
    legend=dict(x=0.45, y=0.05, bgcolor='rgba(26,26,26,0.8)'),
    width=650,
    height=550,
)
fig_roc.write_html(str(IMAGES_PATH / '15_roc_curve_comparison.html'))
fig_roc.show()

## 5.4 Falsos Negativos — Análise crítica

In [17]:
# Comparar falsos negativos (clientes que saem mas o modelo disse que ficam)
fig_fn: go.Figure = go.Figure()

metricas_criticas: list[str] = ['true_positives', 'false_negatives', 'true_negatives', 'false_positives']
labels_metricas: list[str] = ['VP (Churn correto)', 'FN (Churn perdido)', 'VN (Retido correto)', 'FP (Falso alarme)']

fig_fn.add_trace(go.Bar(
    x=labels_metricas,
    y=[metrics_rf[m] for m in metricas_criticas],
    name='Random Forest',
    marker_color='#39FF5A',
    text=[metrics_rf[m] for m in metricas_criticas],
    textposition='outside',
))

fig_fn.add_trace(go.Bar(
    x=labels_metricas,
    y=[metrics_xgb[m] for m in metricas_criticas],
    name='XGBoost',
    marker_color='#004D54',
    text=[metrics_xgb[m] for m in metricas_criticas],
    textposition='outside',
))

fig_fn.update_layout(
    **LAYOUT_PADRAO,
    title='Detalhamento da Matriz de Confusão — RF vs XGBoost',
    yaxis_title='Quantidade de Clientes',
    barmode='group',
)
fig_fn.write_html(str(IMAGES_PATH / '16_confusion_detail_comparison.html'))
fig_fn.show()

### Insights — Comparação

- **Random Forest vence em todas as métricas**: Accuracy (+1.65pp), AUC (+0.0087), Recall (+1.96pp), Precision (+3.72pp).
- **Falsos negativos (FN)** — métrica mais crítica para o negócio: RF perde 117 clientes, XGBoost perde 125. O RF é preferível para **minimizar churn não detectado**.
- **Curva ROC** — ambos bem acima da diagonal aleatória, com desempenho forte a partir de FPR ~0.2.
- **Para produção**: Random Forest como modelo principal. XGBoost poderia ser usado como modelo de ensemble ou com tuning adicional.

---
## 6. Análise de Probabilidade de Churn

In [18]:
# Distribuição das probabilidades de churn previstas pelo melhor modelo
best_model_name: str = 'XGBoost' if auc_xgb >= auc_rf else 'Random Forest'
best_proba: np.ndarray = y_proba_xgb if auc_xgb >= auc_rf else y_proba_rf

df_proba: pd.DataFrame = pd.DataFrame({
    'probabilidade_churn': best_proba,
    'real': y_test.values,
    'status': y_test.map({0: 'Retido', 1: 'Churned'}).values,
})

fig_dist_proba: go.Figure = px.histogram(
    df_proba,
    x='probabilidade_churn',
    color='status',
    nbins=50,
    title=f'Distribuição da Probabilidade de Churn — {best_model_name}',
    labels={'probabilidade_churn': 'Probabilidade de Churn', 'count': 'Frequência', 'status': 'Status Real'},
    color_discrete_map={'Retido': '#39FF5A', 'Churned': '#004D54'},
    barmode='overlay',
    opacity=0.7,
)

fig_dist_proba.add_vline(
    x=0.5, line_dash='dash', line_color='#CCCCCC',
    annotation_text='Threshold 0.5',
    annotation_font_color='#CCCCCC',
)

fig_dist_proba.update_layout(**LAYOUT_PADRAO)
fig_dist_proba.write_html(str(IMAGES_PATH / '17_proba_distribution.html'))
fig_dist_proba.show()

---
## 7. Diagnóstico Final — Previsão de Churn

### 7.1 Modelo Recomendado
| Métrica | Random Forest | XGBoost | Vencedor |
|---|---|---|---|
| Accuracy | **84.85%** | 83.20% | RF |
| ROC-AUC | **0.8798** | 0.8711 | RF |
| Recall (Churn) | **71.25%** | 69.29% | RF |
| Precision (Churn) | **60.92%** | 57.20% | RF |
| Falsos Negativos | **117** | 125 | RF |

**Recomendação:** Random Forest como modelo principal — melhor accuracy, menor taxa de falsos negativos e melhor ROC-AUC.

### 7.2 Features Mais Relevantes
1. **Idade** (`age`) — fator dominante em ambos os modelos, confirmando que clientes 50+ são grupo de risco.
2. **Número de Produtos** (`num_of_products`) — 2ª feature mais importante, refletindo o churn catastrófico em 3+ produtos.
3. **Saldo** (`balance`) — clientes com saldo alto tendem a churnar mais surpreendentemente.
4. **Atividade** (`is_active_member`) — confirma que inatividade é sinal forte de abandono.

### 7.3 Limitações
- **~29% dos churned não são detectados** (117 falsos negativos) — ajuste de threshold para 0.3-0.4 poderia aumentar recall ao custo de mais falsos positivos.
- Dataset relativamente pequeno (10K registros) — modelos mais complexos teriam benefício marginal.
- Sem variáveis temporais (transações, interações) que poderiam melhorar significativamente a previsão.

### 7.4 Recomendações para Stakeholders
1. **Marina Costa**: Usar score de probabilidade para priorizar ações de retenção nos **top 20%** com maior probabilidade.
2. **Gabriel Santos**: Considerar pipeline de retreino mensal e ajuste de threshold conforme custo de FN vs FP.
3. **Sr. Almir**: O modelo identifica **71% dos clientes que abandonariam** — ação proativa pode evitar churn significativo e proteger receita.